<a href="https://colab.research.google.com/github/OscarPasasin/etl-data-pipeline/blob/main/notebooks/Polizas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

https://raw.githubusercontent.com/OscarPasasin/etl-data-pipeline/refs/heads/main/data/raw/polizas.csv

In [13]:
import pandas as pd
import numpy as np

In [14]:
url = "https://raw.githubusercontent.com/OscarPasasin/etl-data-pipeline/refs/heads/main/data/raw/polizas.csv"

df = pd.read_csv(url)

df.head()

,id_poliza,fecha_emision,id_cliente,id_corredor,id_aseguradora,id_tipo_seguro,prima,comision,monto_asegurado
0,1,NaN,184,42,15,2,"829,53",NaN,139253.11
1,2,2026/02/16,2408,35,11,12,NaN,"12,22","27.544,32"
2,3,02-14-25,540,42,4,9,1611.53,"92,05","173,298.36"
3,4,09-01-2026,2821,40,10,5,1866.62,456.99,244461.27
4,5,2026-02-13,945,23,9,11,-,"324,08",123407.75


In [15]:
#Exploracion de datos
df.shape
df.columns
df.info()
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25150 entries, 0 to 25149
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   id_poliza        25150 non-null  int64 
 1   fecha_emision    22739 non-null  object
 2   id_cliente       25150 non-null  int64 
 3   id_corredor      25150 non-null  int64 
 4   id_aseguradora   25150 non-null  int64 
 5   id_tipo_seguro   25150 non-null  int64 
 6   prima            21840 non-null  object
 7   comision         21715 non-null  object
 8   monto_asegurado  21787 non-null  object
dtypes: int64(5), object(4)
memory usage: 1.7+ MB


,0
id_poliza,0
fecha_emision,2411
id_cliente,0
id_corredor,0
id_aseguradora,0
id_tipo_seguro,0
prima,3310
comision,3435
monto_asegurado,3363


In [16]:
#Limpieza de datos
polizas=df.copy()

print(polizas['fecha_emision'].value_counts(dropna=False))

fecha_emision
NaN           2411
18-02-2026      23
07-10-2025      23
27/02/2025      21
27/01/2025      21
              ... 
2025/03/31       3
29-10-2025       3
2025-02-18       3
2025/02/09       2
03-10-2025       2
Name: count, Length: 2081, dtype: int64


In [17]:
polizas['fecha_emision'] = pd.to_datetime(polizas['fecha_emision'], format='mixed', dayfirst=False, errors='coerce')
print(polizas['fecha_emision'].value_counts(dropna=False))

fecha_emision
NaT           2411
2025-03-15      77
2025-11-01      74
2025-07-03      74
2025-09-13      72
              ... 
2026-03-01      17
2026-03-02      17
2026-08-02      17
2026-09-01      16
2026-10-01      15
Name: count, Length: 437, dtype: int64


In [18]:
#Funcion para estandarizar numeros
def limpiar_numero_mixto(v):
    if pd.isna(v): return None

    s = str(v).strip().replace('$', '').replace(' ', '')

    if s in ('', '-', 'N/A', 'nan', 'None'): return None

    if ',' in s and '.' in s:
        s = s.replace('.' if s.find('.') < s.find(',') else ',', '')

    try:
        return float(s.replace(',', '.'))
    except:
        return None

In [19]:
polizas['prima'] = polizas['prima'].apply(limpiar_numero_mixto)
print(polizas['prima'].head(10))

0     829.53
1        NaN
2    1611.53
3    1866.62
4        NaN
5     943.49
6    1400.82
7    1670.56
8        NaN
9     791.38
Name: prima, dtype: float64


In [20]:
polizas['comision'] = polizas['comision'].apply(limpiar_numero_mixto)
print(polizas['prima'].head(10))

0     829.53
1        NaN
2    1611.53
3    1866.62
4        NaN
5     943.49
6    1400.82
7    1670.56
8        NaN
9     791.38
Name: prima, dtype: float64


In [21]:
polizas['monto_asegurado'] = polizas['monto_asegurado'].apply(limpiar_numero_mixto)
print(polizas['monto_asegurado'].head(10))

0    139253.11
1     27544.32
2    173298.36
3    244461.27
4    123407.75
5     71968.43
6    258202.93
7          NaN
8    125804.84
9     20710.00
Name: monto_asegurado, dtype: float64


In [27]:
#Separar datos válidos y rechazados
filtrado_validos = (polizas['fecha_emision'].notna() &\
                    polizas['prima'].notna() & \
                    polizas['comision'].notna() &\
                    polizas['monto_asegurado'].notna())
validos = polizas[filtrado_validos].copy()
rechazados = polizas[~filtrado_validos].copy()

In [28]:
#Motivo de rechazo
def motivo_rechazo(row):
  motivos = []
  if pd.isnull(row['fecha_emision']):
    motivos.append('fecha_emision_vacio')
  if pd.isnull(row['prima']):
    motivos.append('prima_vacio')
  if pd.isnull(row['comision']):
    motivos.append('comision_vacio')
  if pd.isnull(row['monto_asegurado']):
    motivos.append('monto_asegurado_vacio')
  return ','.join(motivos)

rechazados['motivo_rechazo'] = rechazados.apply(motivo_rechazo, axis=1)

In [29]:
validos.to_csv("polizas_curated.csv", index=False)

rechazados.to_csv("polizas_rejects.csv", index=False)